In [ ]:
from pathlib import Path

import seaborn as sns
from bonner.plotting import save_figure
from matplotlib import pyplot as plt
from tqdm.auto import tqdm

from lib.datasets import (
    compute_shared_stimuli,
    filter_by_stimulus,
    nsd,
    split_by_repetition,
)
from lib.spectra import compute_within_individual_spectra, plot_spectra
from lib.utilities import JOURNAL_MATPLOTLIBRC, mathtext_exponent_label

FIGURES_HOME = Path.cwd().parent / "figures"
FIGURES_HOME.mkdir(exist_ok=True, parents=True)

sns.set_theme(context="paper", style="ticks", rc=JOURNAL_MATPLOTLIBRC)

REFERENCE_SUBJECT = 0


In [ ]:
dataset = nsd.load_dataset(
    subject=REFERENCE_SUBJECT,
    preprocessing="fithrf",
    roi="general",
    z_score=True,
)

datasets = split_by_repetition(
    filter_by_stimulus(
        dataset,
        stimuli=compute_shared_stimuli([dataset], n_repetitions=2),
    ),
    n_repetitions=2,
)

spectra = {
    density: compute_within_individual_spectra(
        {0: datasets},
        density=density,
        n_permutations=5_000,
    ).expand_dims(density=[density])
    for density in tqdm([1, 2, 4, 8, 16], desc="density", leave=False)
}

In [ ]:
palette = sns.color_palette("viridis_r", n_colors=len(spectra))

fig, ax = plt.subplots()

for i_density, (density, spectra_) in enumerate(reversed(spectra.items())):
    plot_spectra(
        ax=ax,
        spectra=spectra_,
        hue="density",
        palette=[palette[i_density]],
        # kwargs_significant={"ls": "-", "marker": None, "label": density},
        kwargs_significant={"label": density},
        # kwargs_insignificant={"ls": "-", "marker": None},
        # hide_insignificant=True,
    )
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(left=1)
ax.set_ylim(top=1e-1, bottom=1e-9)
ytick_exponents = list(range(-9, 0))
ax.set_yticks(
    [10**exponent for exponent in ytick_exponents],
    labels=[
        mathtext_exponent_label(exponent) if exponent % 2 == 1 else ""
        for exponent in ytick_exponents
    ],
)
ax.set_title(f"within-subject, subject {1 + REFERENCE_SUBJECT}", pad=10)
ax.set_ylabel("covariance")
ax.set_xlabel("rank")
ax.legend(
    loc="lower left",
    title="bins per decade",
    reverse=True,
)
save_figure(fig, filepath=FIGURES_HOME / "vary-bins-per-decade.pdf")